# Moovio - Πρόβλεψη Κινητικότητας με το μοντέλο Prophet

Παράγονται τρία αποτελέσματα:

1. **Πρόβλεψη συνολικών διαδρομών** για τις επόμενες 14 ημέρες, μαζί με ζώνη αβεβαιότητας 80%.
2. **Πρόβλεψη modal split** για τους τέσσερις βασικούς τρόπους μετακίνησης, με ξεχωριστό μοντέλο ανά τρόπο και κανονικοποίηση ώστε τα μερίδια να αθροίζουν στο 100% κάθε μέρα.
3. **Λίστα ανωμαλιών**: για κάθε τοποθεσία συγκρίνουμε τη σημερινή παρατήρηση με την πρόβλεψη του Prophet και βγάζουμε ένα z-score που δίνει τη σοβαρότητα.

Διαλέξαμε το *Prophet* (Taylor & Letham, 2018) επειδή είναι φτιαγμένο για χρονοσειρές με έντονη εβδομαδιαία περιοδικότητα, ακριβώς ό,τι έχουν τα δεδομένα κινητικότητας, όπου οι εργάσιμες και τα Σαββατοκύριακα διαφέρουν εντελώς. Η τάση και η εποχικότητα του μοντέλου ερμηνεύονται εύκολα, που βοηθάει στην παρουσίαση των αποτελεσμάτων. Το ίδιο μοντέλο τρέχει και στις τρεις εργασίες, ώστε η μεθοδολογία να μένει ενιαία.

Ημερομηνία αναφοράς: **24 Απριλίου 2026**. Ορίζοντας πρόβλεψης: **14 ημέρες**. Δεδομένα εκπαίδευσης: **90 ημέρες**.

## 1. Βιβλιοθήκες

Χρησιμοποιούνται οι βιβλιοθήκες `prophet`, `pandas` και `numpy`.

In [ ]:
!pip install -q prophet pandas numpy scipy

## 2. Δημιουργία της σειράς εκπαίδευσης

Φτιάχνουμε τη σειρά με σταθερό seed ώστε να αναπαράγεται. Είναι η ίδια σειρά που χρησιμοποιεί η πλατφόρμα στο `lib/data/analytics-data.ts` και έχει 90 ημερήσιες τιμές με τρία χαρακτηριστικά:

- ανοδική **τάση** ~18% στο σύνολο της περιόδου,
- εβδομαδιαία **εποχικότητα** με πτώση ~38% τα Σαββατοκύριακα,
- τυχαίο **θόρυβο** για τις φυσικές ημερήσιες διακυμάνσεις.

Σε πραγματικό περιβάλλον η σειρά θα ερχόταν από βάση δεδομένων με μετρήσεις του δικτύου, χωρίς να αλλάξει κάτι άλλο στη ροή.

In [ ]:
import numpy as np
import pandas as pd

REFERENCE_DATE = pd.Timestamp("2026-04-24")
TRAINING_DAYS = 90
FORECAST_DAYS = 14

rng = np.random.default_rng(42)

dates = pd.date_range(end=REFERENCE_DATE, periods=TRAINING_DAYS, freq="D")
dow = dates.dayofweek
is_weekend = (dow >= 5).astype(float)
trend = 1 + np.linspace(0, 0.18, TRAINING_DAYS)
weekend_factor = np.where(is_weekend == 1, 0.62, 1.0)
noise = 0.9 + rng.random(TRAINING_DAYS) * 0.2
trips = (280_000 * trend * weekend_factor * noise).round().astype(int)

history = pd.DataFrame({"ds": dates, "y": trips})
history.head()

## 3. Εκπαίδευση του μοντέλου

Το Prophet βλέπει τη χρονοσειρά $y(t)$ ως άθροισμα τριών συνιστωσών και ενός σφάλματος:

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t,$$

με $g(t)$ την τάση, $s(t)$ την εποχικότητα, $h(t)$ τον όρο αργιών (δεν χρησιμοποιείται εδώ) και $\epsilon_t$ τον θόρυβο.

Κρατάμε μόνο την εβδομαδιαία εποχικότητα, που είναι και το βασικό μοτίβο της σειράς: η ετήσια θέλει πολύ περισσότερα από 90 ημέρες, ενώ η ημερήσια δεν έχει νόημα σε ημερήσιο βήμα. Το διάστημα εμπιστοσύνης ορίζεται στο 80%, όση ακριβώς και η ζώνη που δείχνει το dashboard.

In [ ]:
from prophet import Prophet

model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=False,
    daily_seasonality=False,
    interval_width=0.80,
)
model.fit(history)

## 4. Πρόβλεψη

Επεκτείνουμε τον άξονα κατά 14 ημέρες μετά την ημερομηνία αναφοράς και κρατάμε την κεντρική πρόβλεψη `yhat` μαζί με τα δύο όρια του διαστήματος εμπιστοσύνης (`yhat_lower`, `yhat_upper`).

In [ ]:
future = model.make_future_dataframe(periods=FORECAST_DAYS, freq="D")
forecast = model.predict(future)
forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(20)

## 5. Έλεγχος ακρίβειας

Κρατάμε τις τελευταίες 14 ημέρες ως σύνολο επικύρωσης και εκπαιδεύουμε στις προηγούμενες 76. Στη συνέχεια συγκρίνουμε τις προβλέψεις με τις πραγματικές τιμές με δύο μετρικές: το **MAPE** (σχετικό σφάλμα ως ποσοστό) και το **RMSE** (σφάλμα στις ίδιες μονάδες με τα δεδομένα, διαδρομές).

$$\mathrm{MAPE} = \frac{1}{n}\sum_{i=1}^{n}\left|\frac{y_i - \hat{y}_i}{y_i}\right| \cdot 100\%, \qquad \mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}.$$

In [ ]:
VALIDATION_DAYS = 14
train = history.iloc[:-VALIDATION_DAYS].copy()
valid = history.iloc[-VALIDATION_DAYS:].copy()

val_model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=False,
    daily_seasonality=False,
    interval_width=0.80,
)
val_model.fit(train)
val_future = val_model.make_future_dataframe(periods=VALIDATION_DAYS, freq="D")
val_forecast = val_model.predict(val_future).iloc[-VALIDATION_DAYS:]

y_true = valid["y"].values
y_pred = val_forecast["yhat"].values

mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

print(f"MAPE: {mape:.2f}%")
print(f"RMSE: {rmse:,.0f}")

## 6. Γράφημα πρόβλεψης

Η ιστορική σειρά μαζί με την πρόβλεψη και τη ζώνη εμπιστοσύνης. Η κάθετη γραμμή δείχνει πού σταματούν τα παρατηρούμενα δεδομένα και ξεκινά η πρόβλεψη.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(history["ds"], history["y"], label="Παρατηρούμενες τιμές", color="#0d9488")
ax.plot(forecast["ds"], forecast["yhat"], label="Πρόβλεψη", color="#0284c7", linestyle="--")
ax.fill_between(
    forecast["ds"],
    forecast["yhat_lower"],
    forecast["yhat_upper"],
    color="#0284c7",
    alpha=0.15,
    label="Διάστημα εμπιστοσύνης 80%",
)
ax.axvline(REFERENCE_DATE, color="gray", linestyle=":", alpha=0.6)
ax.legend()
ax.set_title("Ημερήσιες διαδρομές - Παρατηρούμενες τιμές και πρόβλεψη Prophet")
plt.tight_layout()
plt.show()

## 7. Πρόβλεψη μεριδίων μέσων μεταφοράς (modal split)

Η πλατφόρμα δείχνει και πώς αναμένεται να μοιραστεί η κίνηση στους τέσσερις τρόπους: αυτοκίνητο, λεωφορείο, ποδήλατο και πεζή μετακίνηση. Εκπαιδεύουμε ένα ξεχωριστό Prophet ανά τρόπο σε συνθετική σειρά 30 ημερών και μετά κανονικοποιούμε τις τέσσερις προβλέψεις ώστε να αθροίζουν στο 100% κάθε μέρα, που είναι και ο μόνος περιορισμός που έχει νόημα για ένα modal split.

In [ ]:
MODAL_HISTORY_DAYS = 30

modal_dates = pd.date_range(end=REFERENCE_DATE, periods=MODAL_HISTORY_DAYS, freq="D")
modal_rng = np.random.default_rng(404)


def synth_modal_share(base: float, drift: float, noise_std: float) -> np.ndarray:
    progress = np.linspace(0.0, 1.0, MODAL_HISTORY_DAYS)
    weekly = 1.0 + 0.04 * np.sin(2 * np.pi * np.arange(MODAL_HISTORY_DAYS) / 7)
    return base * weekly + drift * progress + modal_rng.normal(0.0, noise_std, MODAL_HISTORY_DAYS)


modal_history = {
    "car": synth_modal_share(46.0, -3.0, 0.7),
    "bus": synth_modal_share(24.0, +1.4, 0.5),
    "bike": synth_modal_share(17.0, +1.6, 0.5),
    "walk": synth_modal_share(13.0, +0.4, 0.4),
}

modal_yhat: dict[str, pd.Series] = {}
for name, series in modal_history.items():
    df = pd.DataFrame({"ds": modal_dates, "y": series})
    m = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.80,
    )
    m.fit(df)
    fut = m.make_future_dataframe(periods=FORECAST_DAYS, freq="D")
    fc = m.predict(fut)
    modal_yhat[name] = fc.set_index("ds")["yhat"]

modal_df = pd.DataFrame(modal_yhat).clip(lower=0.0)
modal_df = modal_df.div(modal_df.sum(axis=1), axis=0) * 100

modal_rows = [
    {
        "date": idx.strftime("%Y-%m-%d"),
        "car": round(float(row["car"]), 1),
        "bus": round(float(row["bus"]), 1),
        "bike": round(float(row["bike"]), 1),
        "walk": round(float(row["walk"]), 1),
        "isFuture": bool(idx > REFERENCE_DATE),
    }
    for idx, row in modal_df.iterrows()
]
modal_rows[-1]

## 8. Ανίχνευση ανωμαλιών μέσω καταλοίπων του Prophet

Η λίστα ανωμαλιών παράγεται από το ίδιο το μοντέλο. Για κάθε τοποθεσία (σταθμός μετρό, αρτηρία, κόμβος) εκπαιδεύουμε ένα Prophet σε συνθετική σειρά 90 ημερών της αντίστοιχης μετρικής, παίρνουμε την πρόβλεψη για την ημέρα αναφοράς και τη συγκρίνουμε με τη σημερινή παρατήρηση.

Μια παρατήρηση είναι ανωμαλία όταν το z-score του καταλοίπου ξεπερνά κάποιο όριο.

$$z = \frac{y_t - \hat{y}_t}{\sigma_t}, \qquad \sigma_t \approx \frac{\hat{y}^{\text{upper}}_t - \hat{y}^{\text{lower}}_t}{2 \cdot z_{0.80}}.$$

Το $\sigma_t$ εκτιμάται από το ότι το διάστημα 80% αντιστοιχεί περίπου σε $\pm 1.282\,\sigma$ για κανονικά κατάλοιπα. Με βάση το $|z|$ ορίζουμε τρία επίπεδα σοβαρότητας, **high** για $|z| \ge 3$, **medium** για $|z| \ge 1.5$ και **low** διαφορετικά.

In [ ]:
from scipy.stats import norm

LOCATIONS = [
    {
        "id": "syntagma",
        "location": "Syntagma Metro",
        "metric": "ridership",
        "unit": "trips/day",
        "baseline": 24500,
        "noise": 1500,
        "weekend_factor": 0.65,
        "today_observed": 18420,
    },
    {
        "id": "kifisias-corridor",
        "location": "Kifisias Ave (Marousi → Kifisia)",
        "metric": "speed",
        "unit": "km/h",
        "baseline": 38.0,
        "noise": 2.5,
        "weekend_factor": 1.10,
        "today_observed": 22.4,
    },
    {
        "id": "monastiraki",
        "location": "Monastiraki Hub",
        "metric": "ridership",
        "unit": "trips/day",
        "baseline": 27000,
        "noise": 1200,
        "weekend_factor": 0.70,
        "today_observed": 31200,
    },
    {
        "id": "attiki-odos",
        "location": "Attiki Odos (Pallini)",
        "metric": "congestion",
        "unit": "%",
        "baseline": 54.0,
        "noise": 4.0,
        "weekend_factor": 0.85,
        "today_observed": 78,
    },
    {
        "id": "piraeus-port",
        "location": "Piraeus Port Stop",
        "metric": "ridership",
        "unit": "trips/day",
        "baseline": 16000,
        "noise": 700,
        "weekend_factor": 0.95,
        "today_observed": 14800,
    },
]

LOC_TRAINING_DAYS = 90
loc_dates = pd.date_range(
    end=REFERENCE_DATE - pd.Timedelta(days=1),
    periods=LOC_TRAINING_DAYS,
    freq="D",
)
is_weekend_loc = (loc_dates.dayofweek >= 5).astype(float)
z_80 = norm.ppf(0.90)


def severity_from_z(z: float) -> str:
    az = abs(z)
    if az >= 3.0:
        return "high"
    if az >= 1.5:
        return "medium"
    return "low"


anomaly_rows = []
for loc in LOCATIONS:
    rng_loc = np.random.default_rng(abs(hash(loc["id"])) % (2**32))
    weekend_mult = np.where(is_weekend_loc == 1, loc["weekend_factor"], 1.0)
    series = loc["baseline"] * weekend_mult + rng_loc.normal(0.0, loc["noise"], LOC_TRAINING_DAYS)

    df = pd.DataFrame({"ds": loc_dates, "y": series})
    m = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.80,
    )
    m.fit(df)
    fut = m.make_future_dataframe(periods=1, freq="D")
    fc = m.predict(fut).iloc[-1]

    expected = float(fc["yhat"])
    sigma = max((float(fc["yhat_upper"]) - float(fc["yhat_lower"])) / (2 * z_80), 1e-6)
    observed = float(loc["today_observed"])
    z = (observed - expected) / sigma

    anomaly_rows.append({
        "id": loc["id"],
        "location": loc["location"],
        "metric": loc["metric"],
        "observed": round(observed, 1),
        "expected": round(expected, 1),
        "unit": loc["unit"],
        "zScore": round(z, 2),
        "severity": severity_from_z(z),
    })

anomaly_rows.sort(key=lambda r: -abs(r["zScore"]))
anomaly_rows

## 9. Εξαγωγή των αποτελεσμάτων

Αποθηκεύουμε όλη την έξοδο (πρόβλεψη διαδρομών, modal split, ανωμαλίες και μετα-δεδομένα) σε ένα JSON, με δομή που ταιριάζει με τους τύπους `TripsForecastPoint`, `ModalForecastPoint`, `AnomalyEntry` και `ModelMetadata` στο `lib/data/predictions-data.ts`. Έτσι η εκπαίδευση μένει ξεχωριστή από το frontend: η εφαρμογή Next.js διαβάζει στατικά την έξοδο, χωρίς server που να τρέχει το μοντέλο σε πραγματικό χρόνο.

In [ ]:
import json

history_lookup = history.set_index("ds")["y"].to_dict()

rows = []
for _, r in forecast.iterrows():
    ds = r["ds"]
    is_future = ds > REFERENCE_DATE
    actual = None if is_future else int(history_lookup.get(ds, None) or 0) or None
    rows.append({
        "date": ds.strftime("%Y-%m-%d"),
        "actual": actual,
        "predicted": int(round(r["yhat"])),
        "band": [int(round(r["yhat_lower"])), int(round(r["yhat_upper"]))],
        "isFuture": bool(is_future),
    })

payload = {
    "tripsForecastData": rows,
    "modalForecastData": modal_rows,
    "anomalyEntries": anomaly_rows,
    "modelMetadata": {
        "algorithm": "Prophet (additive, weekly seasonality)",
        "trainedOn": f"Synthetic mobility series · {TRAINING_DAYS} days",
        "trainingWindowDays": TRAINING_DAYS,
        "horizonDays": FORECAST_DAYS,
        "mape": round(mape, 2),
        "rmse": int(round(rmse)),
        "lastUpdated": REFERENCE_DATE.strftime("%Y-%m-%d"),
    },
}

with open("predictions.json", "w") as f:
    json.dump(payload, f, indent=2)